# Aula 6 - Treino da rede neuronal
## Losses (Slides 3-5)

In [1]:
import numpy as np
from si.neural_networks.losses import LossFunction, MeanSquaredError, BinaryCrossEntropy

# 1. a classe base é abstrata: não se pode criar um objeto dela
try:
    LossFunction()
except TypeError as e:
    print("LossFunction() ->", e)

# 2. gradient check: a derivative tem de bater certo com a derivada numérica da loss
#    derivada numérica de cada valor: (L(p + eps) - L(p - eps)) / (2 * eps)
def derivada_numerica(loss_fn, y_true, y_pred, eps=1e-6):
    numerica = np.zeros_like(y_pred)
    for i in range(y_pred.shape[0]):
        for j in range(y_pred.shape[1]):
            p_mais, p_menos = y_pred.copy(), y_pred.copy()
            p_mais[i, j] += eps
            p_menos[i, j] -= eps
            numerica[i, j] = (loss_fn.loss(y_true, p_mais) - loss_fn.loss(y_true, p_menos)) / (2 * eps)
    return numerica

np.random.seed(0)
y_true = np.random.randint(0, 2, size=(5, 1)).astype(float)  # rótulos 0/1
y_pred = np.random.uniform(0.05, 0.95, size=(5, 1))         # probabilidades longe de 0 e de 1

for loss_fn in [MeanSquaredError(), BinaryCrossEntropy()]:
    analitica = loss_fn.derivative(y_true, y_pred)
    numerica = derivada_numerica(loss_fn, y_true, y_pred)
    print(f"{type(loss_fn).__name__}: loss = {loss_fn.loss(y_true, y_pred):.4f} | "
          f"diferença máx. = {np.abs(analitica - numerica).max():.1e}")

# 3. a BCE castiga muito uma previsão confiante e errada
bce = BinaryCrossEntropy()
y = np.array([[1.0]])
print("y = 1, p = 0.9  ->", round(bce.loss(y, np.array([[0.9]])), 3))
print("y = 1, p = 0.01 ->", round(bce.loss(y, np.array([[0.01]])), 3))
print("y = 1, p = 0    ->", round(bce.loss(y, np.array([[0.0]])), 3), "(sem o clip dava inf)")

LossFunction() -> Can't instantiate abstract class LossFunction with abstract methods derivative, loss
MeanSquaredError: loss = 0.2969 | diferença máx. = 5.3e-11
BinaryCrossEntropy: loss = 4.0772 | diferença máx. = 5.1e-10
y = 1, p = 0.9  -> 0.105
y = 1, p = 0.01 -> 4.605
y = 1, p = 0    -> 34.539 (sem o clip dava inf)


**Conclusão (losses):**
- A `derivative` de cada loss bate certo com a derivada numérica (diferença ~1e-10): é mesmo a derivada da `loss`. É este valor que entra no backward pela última camada.
- MSE para regressão; BCE para classificação binária, com sigmoide na última camada.
- A BCE castiga muito os erros confiantes: com y = 1, p = 0.9 dá 0.105, mas p = 0.01 dá 4.6.
- O `np.clip` evita log(0) = -inf: com p = 0 a loss fica 34.5 em vez de inf.
